# HPD 1 — Ejercicios evaluables

<figure>
<a
href="https://colab.research.google.com/github/Adamychen/m10_quarto/blob/main/notebooks/evaluables/hpd1-evaluables.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open In Colab</figcaption>
</figure>

> **Peso en la nota:** 7.5 % (parte del 30 % de entregas prácticas)
>
> **Plazo:** 7 días tras la sesión presencial.
>
> **Entrega:** Notebook `.ipynb` ejecutado con todas las celdas
> completas. Cada ejercicio especifica qué variable debe contener el
> resultado para la corrección automática.

> **Cómo se corrige**
>
> Cada ejercicio pide que asignes el resultado a una variable con un
> nombre concreto (`eval1_resultado`, `eval2_tabla`, `eval3_recall`,
> `eval4_scores`). El script de corrección ejecutará tu notebook e
> inspeccionará esas variables. **Si la variable no existe o tiene un
> tipo incorrecto, el ejercicio se puntúa como 0.**
>
> Para autoevaluarte antes de entregar:
>
> ``` bash
> python scripts/corregir_hpd1.py tu_notebook.ipynb
> ```

In [1]:
!pip install -q chromadb sentence-transformers fpdf2 matplotlib pandas langchain langchain-community langchain-text-splitters pypdf

In [2]:
import os, tempfile, shutil
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from fpdf import FPDF
from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb

BASE_DIR = tempfile.mkdtemp()
CORPUS_DIR = os.path.join(BASE_DIR, "corpus")
os.makedirs(CORPUS_DIR)

MODEL_NAME = "intfloat/e5-small-v2"
CHUNK_SIZE = 300
CHUNK_OVERLAP = 50

model = SentenceTransformer(MODEL_NAME)
client = chromadb.EphemeralClient()

# Generar corpus de trabajo (3 PDFs que YA viste en la sesión presencial)
class PDF(FPDF):
    pass

def crear_pdf(nombre, titulo, contenido):
    pdf = PDF()
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 16)
    pdf.cell(0, 10, titulo, new_x="LMARGIN", new_y="NEXT")
    pdf.ln(5)
    pdf.set_font("Helvetica", "", 11)
    pdf.multi_cell(0, 6, contenido)
    pdf.output(os.path.join(CORPUS_DIR, nombre))

corpus = {
    "01_transformers.pdf": (
        "Transformers and Attention Mechanisms",
        "The Transformer architecture, introduced by Vaswani et al. in 2017, "
        "revolutionized natural language processing by eliminating recurrences "
        "and relying entirely on attention mechanisms. "
        "The key component is multi-head attention, which allows the model "
        "to simultaneously attend to different parts of the input sequence "
        "from different representation subspaces. Each attention head "
        "computes its own Query, Key, and Value weights through learned "
        "linear projections, and the outputs are concatenated and linearly "
        "projected. This enables capturing both local and global relationships "
        "between tokens without depending on sequential distance. Positional "
        "encoding is added to the input embeddings to preserve word order, "
        "since attention by itself is position-invariant. There are fixed "
        "sinusoidal encodings and learned positional encodings. Transformers "
        "support three main configurations: encoder-only like BERT, decoder-only "
        "like GPT and Llama, and encoder-decoder like T5 and BART. The "
        "decoder-only version is the foundation of modern large language "
        "models such as GPT-4 and Llama 3."
    ),
    "02_embeddings.pdf": (
        "Embeddings and Semantic Vector Representation",
        "Text embeddings are dense vector representations that capture the "
        "semantic meaning of words, phrases, or entire documents in a "
        "continuous vector space of fixed dimensionality. Models such as "
        "Sentence-BERT, E5, and OpenAI text-embedding-3 project texts into "
        "a space where the distance between vectors reflects semantic "
        "similarity, not lexical overlap. The most widely used metric is "
        "cosine similarity, which measures the angle between two normalized "
        "vectors independently of their magnitude. Modern embeddings are "
        "trained with contrastive objectives: maximizing similarity between "
        "semantically equivalent texts while minimizing it between unrelated "
        "ones. Embeddings are the foundation of semantic search and RAG "
        "systems. Models like intfloat/e5-small-v2, with only 33 million "
        "parameters, offer an excellent balance between representation "
        "quality and computational efficiency."
    ),
    "03_rag.pdf": (
        "RAG: Retrieval-Augmented Generation",
        "RAG is an architecture that combines information retrieval systems "
        "with generative models to produce answers grounded in documents. "
        "The typical pipeline consists of three stages: ingestion, retrieval, "
        "and generation. During ingestion, documents are split into chunks, "
        "embeddings are generated, and they are indexed in a vector database "
        "such as Chroma or FAISS. The choice of chunk size and overlap is "
        "critical. During retrieval, the user query is converted into an "
        "embedding and the most similar chunks are retrieved. During "
        "generation, the retrieved chunks are added to the LLM prompt. "
        "Advanced RAG incorporates re-ranking with cross-encoders, query "
        "expansion with HyDE, and RAG-Fusion."
    ),
}

for nombre, (titulo, contenido) in corpus.items():
    crear_pdf(nombre, titulo, contenido)

print(f"Corpus listo en {CORPUS_DIR}")

Corpus listo en /tmp/tmp1wwzwg18/corpus

> **Idioma del corpus y las consultas**
>
> El corpus sintético y las consultas de evaluación están en **inglés**
> porque el modelo `intfloat/e5-small-v2` está entrenado principalmente
> en inglés. Usar texto en español con este modelo degradaría la calidad
> de los embeddings (el tokenizador fragmenta peor y el espacio
> semántico no está alineado). Si tu proyecto requiere español, cambia
> el modelo a uno multilingüe como `intfloat/multilingual-e5-small`.

------------------------------------------------------------------------

## Ejercicio 1 — Añade un cuarto documento (2.5 puntos)

Añade un cuarto PDF al corpus sobre **fine-tuning eficiente con LoRA**.
El contenido debe ocupar al menos 250 palabras y mencionar conceptos
como *low-rank adaptation*, *matrices A y B*, *rango r* y *reducción de
parámetros entrenables*.

Después, construye el pipeline completo (carga → chunking → embeddings →
indexación → búsqueda) y almacena el resultado de la búsqueda para la
query `"How does LoRA reduce the number of trainable parameters?"` con
`k=3`.

| Criterio                                                       | Puntos |
|----------------------------------------------------------------|--------|
| Pipeline completo (carga, chunking, indexación, búsqueda)      | 1.0    |
| Resultado almacenado en `eval1_resultado` con formato correcto | 1.0    |
| Se recuperan exactamente 3 chunks                              | 0.5    |

In [3]:
# ─── Añade aquí el cuarto PDF al diccionario corpus ───
# corpus["04_lora.pdf"] = ("...", "...")
# (Re-ejecuta la celda de generación del corpus tras añadirlo)

# ─── Construye el pipeline completo ───
# 1. Carga los 4 PDFs con PyPDFLoader
# 2. Chunking con RecursiveCharacterTextSplitter
# 3. Indexación en Chroma
# 4. Búsqueda para la query indicada

# ─── Resultado esperado por el corrector ───
# eval1_resultado debe ser un dict: {"ids": [...], "distances": [...], "documents": [...]}
# con los 3 chunks recuperados para "How does LoRA reduce the number of trainable parameters?"

eval1_resultado = None  # ← asigna aquí el resultado de la búsqueda

In [4]:
# ─── Auto-verificación ───
assert isinstance(eval1_resultado, dict), "❌ eval1_resultado debe ser un dict"
assert all(k in eval1_resultado for k in ("ids", "distances", "documents")), \
    "❌ Faltan keys: ids, distances, documents"
assert len(eval1_resultado.get("ids", [])) > 0, "❌ ids está vacío"
assert len(eval1_resultado["ids"][0]) == 3, \
    f"❌ Se esperaban 3 resultados, tienes {len(eval1_resultado['ids'][0])}"
print("✅ Ejercicio 1: formato correcto")

------------------------------------------------------------------------

## Ejercicio 2 — Impacto del chunk size (2.5 puntos)

Ejecuta el pipeline de los 3 primeros documentos (sin el cuarto) con
tres tamaños de chunk distintos: **200, 400 y 800** caracteres (mantén
`chunk_overlap` al 15 % de `chunk_size` en cada caso).

Para cada tamaño, mide: - Número de chunks generados - `recall@1` medio
sobre el ground truth dado

| Criterio                                                 | Puntos |
|----------------------------------------------------------|--------|
| Pipeline ejecutado para 3 tamaños (200, 400, 800)        | 1.0    |
| `eval2_tabla` contiene las 3 filas con claves correctas  | 1.0    |
| Los valores de `n_chunks` y `recall_at_1` son coherentes | 0.5    |

In [5]:
# Ground truth por contenido: invariante a chunk_size y splitter
# Cada query tiene una palabra/frase que DEBE aparecer en el chunk relevante
QRELS = {
    "How does multi-head attention work?":  "multi-head attention",
    "What are sentence embeddings?":        "dense vector representations",
    "What is the Transformer architecture?": "Transformer architecture",
}

def recall_at_k(textos_recuperados, palabra_clave, k):
    """recall@1: 1.0 si palabra_clave aparece en algún texto del top-k."""
    if not palabra_clave:
        return 0.0
    top_k = textos_recuperados[:k]
    return 1.0 if any(palabra_clave.lower() in t.lower() for t in top_k) else 0.0

# ─── Itera sobre chunk_size in [200, 400, 800] ───
# Para cada tamaño:
#   1. Carga los PDFs con PyPDFLoader
#   2. Crea un RecursiveCharacterTextSplitter con ese tamaño
#   3. Indexa en Chroma
#   4. Evalúa recall@1 medio
#   ⚠️ Usa res["documents"][0] (texto) no res["ids"][0] para recall_at_k
#   5. Guarda los resultados

# ─── Resultado esperado por el corrector ───
# eval2_tabla debe ser una lista de dicts:
# [{"chunk_size": 200, "n_chunks": X, "recall_at_1": Y}, ...]

eval2_tabla = []  # ← asigna aquí la lista de resultados

In [6]:
# ─── Auto-verificación ───
assert isinstance(eval2_tabla, list), "❌ eval2_tabla debe ser una lista"
assert len(eval2_tabla) == 3, f"❌ Se esperaban 3 configuraciones, tienes {len(eval2_tabla)}"
for row in eval2_tabla:
    assert all(k in row for k in ("chunk_size", "n_chunks", "recall_at_1")), \
        f"❌ Faltan claves en fila: {row}"
print("✅ Ejercicio 2: formato correcto")

------------------------------------------------------------------------

## Ejercicio 3 — Comparativa de modelos (2.5 puntos)

Cambia el modelo de embeddings a `all-MiniLM-L6-v2` y repite el pipeline
sobre los 3 documentos originales. Calcula el `recall@1` medio con el
mismo ground truth del ejercicio 2.

Escribe **un párrafo** (máximo 150 palabras) justificando qué modelo
elegirías para un proyecto real y por qué. Ten en cuenta el recall
obtenido, el número de parámetros de cada modelo y el benchmark MTEB.

| Criterio                                              | Puntos |
|-------------------------------------------------------|--------|
| Pipeline ejecutado con `all-MiniLM-L6-v2`             | 1.0    |
| `eval3_recall` es un float con el recall@1 medio      | 1.0    |
| Justificación escrita (mín. 10 caracteres, coherente) | 0.5    |

In [7]:
# ─── Pipeline con all-MiniLM-L6-v2 ───
# 1. Carga el modelo alternativo
# 2. Carga PDFs, chunking, indexación, búsqueda
# 3. Calcula recall@1 medio con QRELS (mismo ground truth del Ej. 2)
# ⚠️ Usa res["documents"][0] (texto) no res["ids"][0] para recall_at_k

# ─── Resultado esperado por el corrector ───
# eval3_recall debe ser un float con el recall@1 medio del nuevo modelo.
# eval3_justificacion debe ser un string con el párrafo de justificación.

eval3_recall = None        # ← float
eval3_justificacion = ""   # ← str (máx. 150 palabras)

In [8]:
# ─── Auto-verificación ───
assert isinstance(eval3_recall, (int, float)), "❌ eval3_recall debe ser numérico"
assert isinstance(eval3_justificacion, str), "❌ eval3_justificacion debe ser str"
assert len(eval3_justificacion.strip()) > 10, "❌ Justificación demasiado corta"
print("✅ Ejercicio 3: formato correcto")

------------------------------------------------------------------------

## Ejercicio 4 — Ground truth propio (2.5 puntos)

Usa **solo los 3 primeros PDFs**. Escribe **4 consultas nuevas en
inglés** que no estén en la lista `QRELS`. Para cada consulta, define
**una palabra o frase corta** que el chunk relevante debería contener
(e.g. `"positional encoding"`).

Ejecuta la evaluación `recall@k` para `k=1,3,5` y devuelve la tabla de
resultados.

| Criterio                                     | Puntos |
|----------------------------------------------|--------|
| 4 consultas definidas con su palabra clave   | 1.0    |
| Pipeline ejecutado para k=1, 3, 5            | 1.0    |
| `eval4_scores` es un dict con claves 1, 3, 5 | 0.5    |

In [9]:
# ─── Define tus 4 consultas con palabra_clave ───
mis_queries = {
    # "your query 1?": "keyword from the relevant chunk",
    # "your query 2?": "...",
    # "your query 3?": "...",
    # "your query 4?": "...",
}

# ─── Evalúa recall@k para k = 1, 3, 5 ───
# Usa el mismo pipeline de carga → chunking → indexación → búsqueda.
# Para cada k, calcula la media de recall sobre tus 4 consultas.
# ⚠️ Usa res["documents"][0] (texto) no res["ids"][0] para recall_at_k

# ─── Resultado esperado por el corrector ───
# eval4_scores debe ser un dict: {1: recall@1_medio, 3: recall@3_medio, 5: recall@5_medio}

eval4_scores = {}  # ← dict con claves 1, 3, 5

In [10]:
# ─── Auto-verificación ───
assert isinstance(eval4_scores, dict), "❌ eval4_scores debe ser un dict"
assert all(k in eval4_scores for k in (1, 3, 5)), "❌ Faltan claves 1, 3, 5"
print("✅ Ejercicio 4: formato correcto")